In [1]:
# ===============
# libraries
# ===============

import os, gc, yaml, glob, pickle
import time
import random
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

import warnings
# warnings.filterwarnings('ignore')

# original
import sys
sys.path.append("../src")
from utils.data import sep, show_df, glob_walk, set_seed, save_config_yaml, dict_to_namespace

from datetime import datetime
date = datetime.now().strftime("%Y%m%d")
print(f"TODAY is {date}")

TODAY is 20260111


In [2]:
# =================
# Load data 
# =================
pp_df = pl.read_csv("/mnt/nfs/home/hidebu/study/CSIRO---Image2Biomass-Prediction/data/processed/010_preprocess_from_publicnotebook/csiro_data_split.csv")
my_pp_df = pl.read_csv("/mnt/nfs/home/hidebu/study/CSIRO---Image2Biomass-Prediction/data/processed/001_preprocess_ver04/df_pivot.csv")

sep("pp_df"); show_df(pp_df, 3, True); 
sep("my_pp_df"); show_df(my_pp_df, 3, True); 

pp_df
(357, 1165)


image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,emb1,emb2,emb3,emb4,emb5,emb6,emb7,emb8,emb9,emb10,emb11,emb12,emb13,emb14,emb15,emb16,emb17,emb18,emb19,emb20,emb21,emb22,emb23,emb24,emb25,emb26,…,emb1118,emb1119,emb1120,emb1121,emb1122,emb1123,emb1124,emb1125,emb1126,emb1127,emb1128,emb1129,emb1130,emb1131,emb1132,emb1133,emb1134,emb1135,emb1136,emb1137,emb1138,emb1139,emb1140,emb1141,emb1142,emb1143,emb1144,emb1145,emb1146,emb1147,emb1148,emb1149,emb1150,emb1151,emb1152,Sampling_Date_Month,fold
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""/kaggle/input/csiro-biomass/tr…","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,0.0,31.9984,16.2751,48.2735,16.275,0.030252,-0.330361,0.105993,-0.498448,0.165929,-0.333561,0.302322,0.324378,0.251653,-0.21362,0.6612024,0.5386001,-0.085385,0.067472,-0.118739,-0.013862,0.092023,0.035291,-0.098196,0.6739824,-0.114674,0.2825343,-0.13526,-0.099973,-0.112377,-0.022378,…,-1.642261,0.046456,0.083476,-0.012412,0.161593,-0.101197,-0.28399,0.199676,-0.080096,0.202713,0.204874,-0.229584,0.050028,-0.48556,0.3365674,0.237784,-0.00326,0.12213,-1.141352,0.576388,-0.146477,0.4755531,0.283804,0.28167,0.245712,0.157092,-0.105552,-0.791377,-0.783494,-0.078441,-0.333965,-0.250307,-0.507553,-0.500009,-0.611312,9,1
"""/kaggle/input/csiro-biomass/tr…","""2015/4/1""","""NSW""","""Lucerne""",0.55,16.0,0.0,0.0,7.6,7.6,7.6,0.136716,0.071181,0.205273,-0.152278,-0.055866,-0.702051,0.0919175,0.6592206,-0.695861,0.113913,0.850891,0.179118,-0.092981,-0.193506,0.090837,-0.116835,0.08469,-0.163651,-0.522117,0.94602,-0.031056,-0.02962,-0.361154,-0.138392,-0.189874,0.5253662,…,-1.595927,0.229376,0.3155437,-0.11973,-0.047137,-0.013973,-0.379212,-0.024193,-0.022275,0.2239867,-0.034441,-0.331555,-0.109393,-0.256027,0.7351272,0.118433,-0.181392,-0.181896,-1.062311,0.9049293,0.160504,0.053374,0.020289,0.071955,-0.475449,0.050482,0.723569,0.075425,-0.407983,0.421373,-0.230122,0.460918,-0.306598,0.045089,-0.035254,4,4
"""/kaggle/input/csiro-biomass/tr…","""2015/9/1""","""WA""","""SubcloverDalkeith""",0.38,1.0,6.05,0.0,0.0,6.05,6.05,0.346141,0.055141,0.127397,-0.256826,-0.056476,-0.347492,0.055052,0.684451,-0.076869,0.066209,0.8313518,0.233694,-0.27879,0.074817,-0.146367,0.012318,-0.13054,-0.12892,-0.287671,0.8735727,0.032198,0.30496,-0.154226,-0.255326,0.028173,0.218366,…,-1.519841,-0.198207,0.5365758,-0.106453,-0.123023,-0.141142,-0.487149,-0.095118,0.1662864,0.596002,-0.08511,-0.31645,-0.021583,-0.449098,0.080767,0.092537,0.121547,-0.004668,-1.142918,0.719163,-0.00991,0.276,0.09716,0.020289,-0.245311,0.167905,0.351724,-0.032907,-0.75382,0.151818,-0.371174,0.014008,-0.562714,-0.103918,-0.623414,9,2


image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,emb1,emb2,emb3,emb4,emb5,emb6,emb7,emb8,emb9,emb10,emb11,emb12,emb13,emb14,emb15,emb16,emb17,emb18,emb19,emb20,emb21,emb22,emb23,emb24,emb25,emb26,…,emb1118,emb1119,emb1120,emb1121,emb1122,emb1123,emb1124,emb1125,emb1126,emb1127,emb1128,emb1129,emb1130,emb1131,emb1132,emb1133,emb1134,emb1135,emb1136,emb1137,emb1138,emb1139,emb1140,emb1141,emb1142,emb1143,emb1144,emb1145,emb1146,emb1147,emb1148,emb1149,emb1150,emb1151,emb1152,Sampling_Date_Month,fold
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""/kaggle/input/csiro-biomass/tr…","""2015/2/24""","""NSW""","""Phalaris""",0.69,29.0,0.0,1.1457,91.6543,92.8,91.6543,-0.090312,0.049795,0.102914,-0.438652,0.3151303,-0.141164,0.2339738,0.403195,-0.073545,-0.118893,0.5497857,0.5012264,-0.112748,0.077709,-0.246067,0.030188,-0.083354,-0.085621,-0.553986,0.7898103,0.07595,0.001571,-0.067461,-0.142621,-0.338257,0.163694,…,-1.743661,-0.031908,0.342105,-0.132671,0.223296,-0.202892,-0.488466,0.108072,-0.059815,0.502498,0.1590413,-0.286574,0.190705,-0.351209,0.270231,0.079972,0.083225,0.159015,-0.962397,0.6713444,-0.291511,0.5703396,0.459558,0.0873235,0.048533,0.150714,0.3538905,-0.48357,-0.809283,-0.318594,-0.526285,0.117585,-0.743469,-0.153629,-0.292729,2,0
"""/kaggle/input/csiro-biomass/tr…","""2015/7/8""","""WA""","""Clover""",0.74,2.0,32.3575,0.0,2.0325,34.39,34.39,0.319788,0.062742,-0.021059,-0.229915,0.206869,0.238602,0.249325,0.9735599,-0.011513,0.1484,0.596072,0.01743,-0.383654,0.086974,-0.342557,0.118335,-0.160819,-0.225957,-0.426007,0.881997,0.050235,0.3679313,-0.107315,-0.001144,0.3170241,0.094059,…,-1.372762,-0.3053,0.312477,0.177984,-0.35032,-0.173597,-0.042795,0.076583,0.151379,0.482448,-0.023642,-0.37168,-0.209928,-0.128563,0.6171634,-0.157704,0.363718,-0.149127,-0.957808,0.8089767,-0.167748,0.517163,0.2233491,-0.126126,0.082291,0.173808,0.3497029,0.072566,-0.666046,0.045908,-0.534787,-0.014465,-0.420235,-0.309601,-0.614334,7,4
"""/kaggle/input/csiro-biomass/tr…","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,0.0,0.0,40.94,40.94,40.94,-0.119426,-0.141372,0.041381,-0.669446,0.286613,-0.099342,-0.00842,0.570014,0.083906,-0.111183,0.005831,0.6866863,-0.287848,-0.107117,-0.162093,0.115433,-0.144447,0.005293,0.224264,0.395968,0.088264,0.0636,-0.13084,-0.24825,-0.233221,-0.034981,…,-1.541735,0.13539,-0.008262,-0.150317,0.049359,-0.17216,-0.255835,0.264173,0.00441,0.173271,-0.012847,-0.37382,0.105704,-0.422378,0.195751,-0.03934,-0.266474,0.261408,-1.171688,0.752469,-0.065443,0.354037,0.396801,0.493451,0.106962,0.372209,0.086259,-0.761338,-0.631423,0.010757,-0.201014,0.124727,-0.734944,-0.449513,-0.041214,9,2


my_pp_df
(357, 36)


image_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,date,year,month,day,weekday,year_month,weekday_name,is_BarleyGrass,is_Barleygrass,is_Bromegrass,is_Capeweed,is_Clover,is_CrumbWeed,is_Fescue,is_Lucerne,is_Mixed,is_Phalaris,is_Ryegrass,is_SilverGrass,is_SpearGrass,is_SubcloverDalkeith,is_SubcloverLosa,is_WhiteClover,Fold
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,str,i64,i64,i64,i64,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,i64
"""ID1011485656""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,0.0,31.9984,16.2751,48.2735,16.275,"""2015-09-04T00:00:00.000""",2015,9,4,5,"""2015-9""","""Fri""",false,false,false,false,true,false,false,false,false,false,true,false,false,false,false,false,2
"""ID1012260530""","""train/ID1012260530.jpg""","""2015/4/1""","""NSW""","""Lucerne""",0.55,16.0,0.0,0.0,7.6,7.6,7.6,"""2015-04-01T00:00:00.000""",2015,4,1,3,"""2015-4""","""Wed""",false,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,2
"""ID1025234388""","""train/ID1025234388.jpg""","""2015/9/1""","""WA""","""SubcloverDalkeith""",0.38,1.0,6.05,0.0,0.0,6.05,6.05,"""2015-09-01T00:00:00.000""",2015,9,1,2,"""2015-9""","""Tue""",false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,1


image_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,date,year,month,day,weekday,year_month,weekday_name,is_BarleyGrass,is_Barleygrass,is_Bromegrass,is_Capeweed,is_Clover,is_CrumbWeed,is_Fescue,is_Lucerne,is_Mixed,is_Phalaris,is_Ryegrass,is_SilverGrass,is_SpearGrass,is_SubcloverDalkeith,is_SubcloverLosa,is_WhiteClover,Fold
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,str,i64,i64,i64,i64,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,i64
"""ID980538882""","""train/ID980538882.jpg""","""2015/2/24""","""NSW""","""Phalaris""",0.69,29.0,0.0,1.1457,91.6543,92.8,91.6543,"""2015-02-24T00:00:00.000""",2015,2,24,2,"""2015-2""","""Tue""",false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,0
"""ID980878870""","""train/ID980878870.jpg""","""2015/7/8""","""WA""","""Clover""",0.74,2.0,32.3575,0.0,2.0325,34.39,34.39,"""2015-07-08T00:00:00.000""",2015,7,8,3,"""2015-7""","""Wed""",false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,0
"""ID983582017""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,0.0,0.0,40.94,40.94,40.94,"""2015-09-01T00:00:00.000""",2015,9,1,2,"""2015-9""","""Tue""",false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,1


In [4]:
pp_df["image_path"].to_list()[:3]

['/kaggle/input/csiro-biomass/train/ID1011485656.jpg',
 '/kaggle/input/csiro-biomass/train/ID1012260530.jpg',
 '/kaggle/input/csiro-biomass/train/ID1025234388.jpg']

In [7]:
print(my_pp_df.columns)

['image_id', 'image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g', 'date', 'year', 'month', 'day', 'weekday', 'year_month', 'weekday_name', 'is_BarleyGrass', 'is_Barleygrass', 'is_Bromegrass', 'is_Capeweed', 'is_Clover', 'is_CrumbWeed', 'is_Fescue', 'is_Lucerne', 'is_Mixed', 'is_Phalaris', 'is_Ryegrass', 'is_SilverGrass', 'is_SpearGrass', 'is_SubcloverDalkeith', 'is_SubcloverLosa', 'is_WhiteClover', 'Fold']


In [8]:
select_cols = ['image_id', 'image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g', 'fold']

In [10]:
pp_df = (
    pp_df.with_columns([
        pl.col("image_path").map_elements(lambda x: os.path.basename(x).split(".")[0], return_dtype=pl.Utf8).alias("image_id"),
        pl.col("image_path").map_elements(lambda x: os.path.basename(x), return_dtype=pl.Utf8).alias("image_path"),
    ])
    .select(select_cols)
    .rename({"fold":"Fold"})
    )

pp_df

image_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,Fold
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,i64
"""ID1011485656""","""ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,0.0,31.9984,16.2751,48.2735,16.275,1
"""ID1012260530""","""ID1012260530.jpg""","""2015/4/1""","""NSW""","""Lucerne""",0.55,16.0,0.0,0.0,7.6,7.6,7.6,4
"""ID1025234388""","""ID1025234388.jpg""","""2015/9/1""","""WA""","""SubcloverDalkeith""",0.38,1.0,6.05,0.0,0.0,6.05,6.05,2
"""ID1028611175""","""ID1028611175.jpg""","""2015/5/18""","""Tas""","""Ryegrass""",0.66,5.0,0.0,30.9703,24.2376,55.2079,24.2376,1
"""ID1035947949""","""ID1035947949.jpg""","""2015/9/11""","""Tas""","""Ryegrass""",0.54,3.5,0.4343,23.2239,10.5261,34.1844,10.9605,1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""ID975115267""","""ID975115267.jpg""","""2015/7/8""","""WA""","""Clover""",0.73,3.0,40.03,0.0,0.8,40.83,40.83,4
"""ID978026131""","""ID978026131.jpg""","""2015/9/4""","""Tas""","""Clover""",0.83,3.1667,24.6445,4.1948,12.0601,40.8994,36.7046,1
"""ID980538882""","""ID980538882.jpg""","""2015/2/24""","""NSW""","""Phalaris""",0.69,29.0,0.0,1.1457,91.6543,92.8,91.6543,0


In [ ]:
pp_df.write_csv("/mnt/nfs/home/hidebu/study/CSIRO---Image2Biomass-Prediction/data/processed/010_preprocess_from_publicnotebook/df_pivot.csv")